<a href="https://colab.research.google.com/github/GhalaAwd/Computer-Vision-Capstone-Project/blob/main/notebooks/01_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics -q

In [2]:
from ultralytics import YOLO

model = YOLO("yolo11m-pose.pt")

image_path = "https://ultralytics.com/images/bus.jpg"

results = model.predict(
    source=image_path,
    save=True,
    project="outputs",
    name="image_test",
    exist_ok=True,
)

print(f"Image inference done. Detected {len(results[0].keypoints)} person(s).")
print(f"Annotated output saved under: outputs/image_test/")


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 /content/bus.jpg: 640x480 4 persons, 29.7ms
Speed: 63.1ms preprocess, 29.7ms inference, 38.0ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/runs/pose/outputs/image_test
Image inference done. Detected 4 person(s).
Annotated output saved under: outputs/image_test/


In [5]:
import os
for root, dirs, files in os.walk("/content/runs"):
    for f in files:
        print(os.path.join(root, f))

/content/runs/pose/outputs/image_test/bus.jpg


In [3]:
# Step 4: Run pose inference on the 3 presentation clips (RAM-safe version)
video_clips = [
    "/content/clip1_presenter.mp4",
    "/content/clip2_presenter.mp4",
    "/content/clip3_presenter.mp4",
]

for clip_path in video_clips:
    clip_name = clip_path.split("/")[-1].replace(".mp4", "")

    # stream=True makes this a generator: results are produced and
    # released one frame at a time instead of all being held in RAM
    # at once. We must loop through it to actually process the video --
    # just calling predict() without iterating won't run anything.
    results_generator = model.predict(
        source=clip_path,
        save=True,
        project="outputs",
        name=f"video_test_{clip_name}",
        exist_ok=True,
        stream=True,
    )

    frame_count = 0
    for r in results_generator:
        frame_count += 1  # touching each result is what drives the loop forward

    print(f"Done: {clip_name} -- processed {frame_count} frames.")
    print(f"Annotated output saved under: outputs/video_test_{clip_name}/")
    print("---")


video 1/1 (frame 1/408) /content/clip1_presenter.mp4: 384x640 9 persons, 57.7ms
video 1/1 (frame 2/408) /content/clip1_presenter.mp4: 384x640 7 persons, 25.4ms
video 1/1 (frame 3/408) /content/clip1_presenter.mp4: 384x640 7 persons, 25.4ms
video 1/1 (frame 4/408) /content/clip1_presenter.mp4: 384x640 8 persons, 25.4ms
video 1/1 (frame 5/408) /content/clip1_presenter.mp4: 384x640 8 persons, 68.9ms
video 1/1 (frame 6/408) /content/clip1_presenter.mp4: 384x640 7 persons, 46.9ms
video 1/1 (frame 7/408) /content/clip1_presenter.mp4: 384x640 7 persons, 35.2ms
video 1/1 (frame 8/408) /content/clip1_presenter.mp4: 384x640 7 persons, 35.4ms
video 1/1 (frame 9/408) /content/clip1_presenter.mp4: 384x640 8 persons, 76.2ms
video 1/1 (frame 10/408) /content/clip1_presenter.mp4: 384x640 8 persons, 59.0ms
video 1/1 (frame 11/408) /content/clip1_presenter.mp4: 384x640 8 persons, 47.9ms
video 1/1 (frame 12/408) /content/clip1_presenter.mp4: 384x640 7 persons, 64.6ms
video 1/1 (frame 13/408) /content/cl

In [6]:
"""
pose_utils.py

Core contract for the MentorVision pipeline. Runs YOLO pose estimation
over a video and returns per-frame COCO-17 keypoints as a DataFrame,
filtered to the main presenter only (privacy rule -- audience dropped).

Everyone else on the team (M2, M3) builds on top of the CSV this
produces, so don't change the column names without telling the team.

COCO-17 keypoint index reference (used by the kp_idx column):
 0 nose        1 left_eye     2 right_eye    3 left_ear     4 right_ear
 5 left_shoulder  6 right_shoulder  7 left_elbow   8 right_elbow
 9 left_wrist     10 right_wrist    11 left_hip     12 right_hip
13 left_knee      14 right_knee     15 left_ankle   16 right_ankle
"""

import pandas as pd
from ultralytics import YOLO


def extract_presenter_keypoints(video_path: str, model_path: str = "yolo11m-pose.pt") -> pd.DataFrame:
    """
    Run pose estimation over a video and return per-frame keypoints
    for the main presenter only (audience filtered out).

    Uses stream=True so results are processed one frame at a time
    instead of held in RAM all at once -- without this, longer videos
    or crowded clips (many people per frame) can crash the runtime.

    Parameters
    ----------
    video_path : str
        Path to the input video file.
    model_path : str
        Path or name of the YOLO pose model weights.

    Returns
    -------
    pd.DataFrame
        Columns: frame, person_id, kp_idx, kp_x, kp_y, conf
        One row per keypoint per frame, presenter only.
    """
    model = YOLO(model_path)

    results_generator = model.predict(
        source=video_path,
        stream=True,
    )

    rows = []
    for frame_idx, result in enumerate(results_generator):
        if result.keypoints is None or result.boxes is None or len(result.boxes) == 0:
            continue

        # --- Privacy rule: keep only the main presenter ---
        # Heuristic: the presenter is the person with the largest
        # bounding-box area (closest to camera / most central on stage).
        # Your clips have 5-11 people per frame (audience visible), so
        # this filter is essential -- without it the CSV would mix in
        # random audience members.
        boxes = result.boxes.xyxy.cpu().numpy()  # [N, 4] -> x1,y1,x2,y2
        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        presenter_idx = int(areas.argmax())

        person_id = "presenter"  # single fixed label per video

        kpts = result.keypoints.xy[presenter_idx].cpu().numpy()   # [17, 2]
        confs = result.keypoints.conf[presenter_idx].cpu().numpy()  # [17]

        for kp_idx in range(kpts.shape[0]):
            rows.append({
                "frame": frame_idx,
                "person_id": person_id,
                "kp_idx": kp_idx,
                "kp_x": float(kpts[kp_idx, 0]),
                "kp_y": float(kpts[kp_idx, 1]),
                "conf": float(confs[kp_idx]),
            })

    df = pd.DataFrame(rows, columns=["frame", "person_id", "kp_idx", "kp_x", "kp_y", "conf"])
    return df


if __name__ == "__main__":
    # Quick manual test on all 3 clips -- adjust paths if needed.
    clips = [
        "/content/clip1_presenter.mp4",
        "/content/clip2_presenter.mp4",
        "/content/clip3_presenter.mp4",
    ]

    all_dfs = []
    for clip_path in clips:
        clip_name = clip_path.split("/")[-1].replace(".mp4", "")
        print(f"Processing {clip_name}...")
        df = extract_presenter_keypoints(clip_path)
        df["clip"] = clip_name  # tag which clip each row came from
        all_dfs.append(df)
        print(f"  -> {len(df)} keypoint rows ({df['frame'].nunique()} frames)")

    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv("outputs/presenter_keypoints.csv", index=False)
    print(f"\nSaved {len(combined)} total keypoint rows to outputs/presenter_keypoints.csv")
    print(combined.head(20))

Processing clip1_presenter...

video 1/1 (frame 1/408) /content/clip1_presenter.mp4: 384x640 9 persons, 25.3ms
video 1/1 (frame 2/408) /content/clip1_presenter.mp4: 384x640 7 persons, 25.4ms
video 1/1 (frame 3/408) /content/clip1_presenter.mp4: 384x640 7 persons, 25.4ms
video 1/1 (frame 4/408) /content/clip1_presenter.mp4: 384x640 8 persons, 29.3ms
video 1/1 (frame 5/408) /content/clip1_presenter.mp4: 384x640 8 persons, 28.9ms
video 1/1 (frame 6/408) /content/clip1_presenter.mp4: 384x640 7 persons, 25.4ms
video 1/1 (frame 7/408) /content/clip1_presenter.mp4: 384x640 7 persons, 34.3ms
video 1/1 (frame 8/408) /content/clip1_presenter.mp4: 384x640 7 persons, 20.7ms
video 1/1 (frame 9/408) /content/clip1_presenter.mp4: 384x640 8 persons, 20.6ms
video 1/1 (frame 10/408) /content/clip1_presenter.mp4: 384x640 8 persons, 20.6ms
video 1/1 (frame 11/408) /content/clip1_presenter.mp4: 384x640 8 persons, 20.6ms
video 1/1 (frame 12/408) /content/clip1_presenter.mp4: 384x640 7 persons, 18.8ms
video 

In [7]:
df = extract_presenter_keypoints("/content/clip1_presenter.mp4")
df.to_csv("outputs/clip1_keypoints.csv", index=False)
print(f"{len(df)} keypoint rows, {df['frame'].nunique()} frames")
df.head(20)


video 1/1 (frame 1/408) /content/clip1_presenter.mp4: 384x640 9 persons, 25.3ms
video 1/1 (frame 2/408) /content/clip1_presenter.mp4: 384x640 7 persons, 25.4ms
video 1/1 (frame 3/408) /content/clip1_presenter.mp4: 384x640 7 persons, 25.4ms
video 1/1 (frame 4/408) /content/clip1_presenter.mp4: 384x640 8 persons, 25.4ms
video 1/1 (frame 5/408) /content/clip1_presenter.mp4: 384x640 8 persons, 25.4ms
video 1/1 (frame 6/408) /content/clip1_presenter.mp4: 384x640 7 persons, 25.4ms
video 1/1 (frame 7/408) /content/clip1_presenter.mp4: 384x640 7 persons, 19.5ms
video 1/1 (frame 8/408) /content/clip1_presenter.mp4: 384x640 7 persons, 18.8ms
video 1/1 (frame 9/408) /content/clip1_presenter.mp4: 384x640 8 persons, 22.3ms
video 1/1 (frame 10/408) /content/clip1_presenter.mp4: 384x640 8 persons, 20.7ms
video 1/1 (frame 11/408) /content/clip1_presenter.mp4: 384x640 8 persons, 21.4ms
video 1/1 (frame 12/408) /content/clip1_presenter.mp4: 384x640 7 persons, 19.4ms
video 1/1 (frame 13/408) /content/cl

,frame,person_id,kp_idx,kp_x,kp_y,conf
0,0,presenter,0,559.066895,912.238037,0.066104
1,0,presenter,1,519.914917,878.929199,0.030516
2,0,presenter,2,592.353577,857.370789,0.049258
3,0,presenter,3,416.701019,932.883606,0.494566
4,0,presenter,4,694.454285,886.178528,0.758071
5,0,presenter,5,327.910583,1266.726807,0.696931
6,0,presenter,6,908.807007,1148.929077,0.829968
7,0,presenter,7,330.862640,1440.000000,0.051634
8,0,presenter,8,1005.541260,1356.976440,0.188373
9,0,presenter,9,451.922119,1355.771362,0.020989


In [5]:
import os
os.makedirs("outputs", exist_ok=True)

combined.to_csv("outputs/presenter_keypoints.csv", index=False)
print(f"Saved {len(combined)} total keypoint rows to outputs/presenter_keypoints.csv")
print(combined.head(20))

Saved 16167 total keypoint rows to outputs/presenter_keypoints.csv
    frame  person_id  kp_idx         kp_x         kp_y      conf  \
0       0  presenter       0   559.066895   912.238037  0.066104   
1       0  presenter       1   519.914917   878.929199  0.030516   
2       0  presenter       2   592.353577   857.370789  0.049258   
3       0  presenter       3   416.701019   932.883606  0.494566   
4       0  presenter       4   694.454285   886.178528  0.758071   
5       0  presenter       5   327.910583  1266.726807  0.696931   
6       0  presenter       6   908.807007  1148.929077  0.829968   
7       0  presenter       7   330.862640  1440.000000  0.051634   
8       0  presenter       8  1005.541260  1356.976440  0.188373   
9       0  presenter       9   451.922119  1355.771362  0.020989   
10      0  presenter      10   826.974915  1259.709106  0.069551   
11      0  presenter      11   468.330200  1440.000000  0.030930   
12      0  presenter      12   891.874695  1440.0